In [1]:
from util import log_flush
from data_loader.loader_util import get_dataloader
from teacher_model import TeacherModel
from student_model import StudentModel
from config import experience_constructor_meta
import datetime
from experience_builder import ExperienceBuilder
from tester import Tester
from util import log_flush
from model.model_loader import load_model
import pandas as pd
from experiment_analyzer.single_run_analyzer import SingleRunAnalyzer

In [2]:
model_path = "/root/autodl-tmp/Meta-Llama-3-8B-Instruct"
model = load_model("local", model_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


In [3]:
# hyper parameter
subject = "biology"
dataset="MMLU"
subset = "high_school_biology"

In [4]:
dataloader = get_dataloader(dataset, subset)

In [5]:
train_set, test_set = dataloader.load_train_test_set(1, random_shuffle=False)

In [6]:
train_set[0]

{'question': 'Which of the following is not a way to form recombinant DNA?',
 'truth': 'Translation',
 'options': array(['Translation', 'Conjugation', 'Specialized transduction',
        'Transformation'], dtype=object),
 'reasoning': ''}

In [7]:
teacher_model = TeacherModel(subject=subject, model=model, trace_score=True)
student_model = StudentModel(model=model)

In [8]:
teacher_model.load_question(train_set[0])

In [9]:
teacher_model.question

'Which of the following is not a way to form recombinant DNA?'

In [10]:
teacher_model.clean_options

['translation', 'conjugation', 'specialized transduction', 'transformation']

In [11]:
teacher_model.get_context()

''

In [12]:
prompt = student_model.pm.generate_llama_percise_question_prompt(teacher_model.question, teacher_model.clean_options, teacher_model.get_context())
student_model.pm.generate_llama_percise_question_prompt(teacher_model.question, teacher_model.clean_options, teacher_model.get_context())

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n                You are a helpful AI assistant. The final answer should be as short and percise as possible. Choose the answer from the options. Answer the final answer in the end after #### <|eot_id|>\n                <|start_header_id|>user<|end_header_id|>\nOptions: '['translation', 'conjugation', 'specialized transduction', 'transformation']'\n\nRequest: Which of the following is not a way to form recombinant DNA?<|eot_id|>\n                <|start_header_id|>assistant<|end_header_id|>"

In [13]:
ans = student_model.model.clean_pipe(prompt)
print(ans)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>
                You are a helpful AI assistant. The final answer should be as short and percise as possible. Choose the answer from the options. Answer the final answer in the end after #### <|eot_id|>
                <|start_header_id|>user<|end_header_id|>
Options: '['translation', 'conjugation', 'specialized transduction', 'transformation']'

Request: Which of the following is not a way to form recombinant DNA?<|eot_id|>
                <|start_header_id|>assistant<|end_header_id|>

#### transformation


In [14]:
target_option = student_model.pm.retrieve_llama_percise_answer(ans)
print(target_option)

transformation


In [43]:
confidence_prompt = student_model.pm.generate_llama_percise_confidence_prompt(teacher_model.question, "translation", teacher_model.clean_options, teacher_model.get_context())
print(confidence_prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>
                You are a helpful AI assistant. The final answer should be as short and percise as possible. Choose the answer from the options. Answer the final answer in the end after #### <|eot_id|>
                <|start_header_id|>user<|end_header_id|>
Options: '['translation', 'conjugation', 'specialized transduction', 'transformation']'

Request: What is your confidence that 'translation' is correct answer to question: Which of the following is not a way to form recombinant DNA?<|eot_id|>
                <|start_header_id|>assistant<|end_header_id|>


In [44]:
result = student_model.model.clean_pipe(confidence_prompt)
print(result)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>
                You are a helpful AI assistant. The final answer should be as short and percise as possible. Choose the answer from the options. Answer the final answer in the end after #### <|eot_id|>
                <|start_header_id|>user<|end_header_id|>
Options: '['translation', 'conjugation', 'specialized transduction', 'transformation']'

Request: What is your confidence that 'translation' is correct answer to question: Which of the following is not a way to form recombinant DNA?<|eot_id|>
                <|start_header_id|>assistant<|end_header_id|>

#### 90%

The correct answer is 'translation'.


In [45]:
student_model.pm.retrieve_llama_percise_confidence(result)

Final result: 90


90

In [35]:
from util import *

In [37]:
keyword = "####"
cur = cut_last_appearance(result, keyword)
cur

'80\n\ntransformation'

In [38]:
"%" in cur

False

In [39]:
"\n" in cur

True

In [41]:
cur = cut_last_appearance_beforewards(cur, "\n")

In [42]:
cur

'80'

In [43]:
cur.isnumeric()

True